# EDA — E-Commerce Fraud Data (`Fraud_Data.csv`)

**Objective:** Understand the structure, distributions, and fraud patterns in the e-commerce transaction dataset. This notebook covers:
1. Data loading and basic profiling
2. Univariate distributions
3. Bivariate analysis vs. fraud target
4. Class imbalance quantification
5. Geolocation analysis (IP → Country)
6. Cleaning: duplicates, missing values, dtypes

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_fraud_data, load_ip_country, basic_info
from src.preprocessing import drop_duplicates, handle_missing, merge_ip_country

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')
print('Setup complete')

## 1. Load Data

In [ ]:
fraud_df = load_fraud_data()
ip_df = load_ip_country()
basic_info(fraud_df, 'Fraud_Data')
fraud_df.head()

## 2. Class Imbalance

In [ ]:
class_counts = fraud_df['class'].value_counts()
class_pct = fraud_df['class'].value_counts(normalize=True) * 100

print('Class Distribution:')
print(pd.DataFrame({'count': class_counts, 'pct': class_pct.round(2)}))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
class_counts.plot.bar(ax=axes[0], color=['steelblue','tomato'], edgecolor='white')
axes[0].set_title('Transaction Count by Class')
axes[0].set_xticklabels(['Legit (0)', 'Fraud (1)'], rotation=0)
axes[0].set_ylabel('Count')

axes[1].pie(class_counts, labels=['Legit', 'Fraud'], autopct='%1.2f%%',
            colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Class Proportion')
plt.tight_layout()
plt.show()

## 3. Univariate Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(fraud_df['purchase_value'], bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Purchase Value Distribution')

sns.histplot(fraud_df['age'], bins=30, ax=axes[1], color='slateblue')
axes[1].set_title('Age Distribution')

fraud_df['source'].value_counts().plot.bar(ax=axes[2], color='teal', edgecolor='white')
axes[2].set_title('Traffic Source')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fraud_df['browser'].value_counts().plot.bar(ax=axes[0], color='coral', edgecolor='white')
axes[0].set_title('Browser Distribution')
axes[0].tick_params(axis='x', rotation=30)

fraud_df['sex'].value_counts().plot.bar(ax=axes[1], color=['royalblue', 'orchid'])
axes[1].set_title('Sex Distribution')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 4. Bivariate Analysis vs. Fraud

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.boxplot(x='class', y='purchase_value', data=fraud_df, ax=axes[0],
            palette={0: 'steelblue', 1: 'tomato'})
axes[0].set_xticklabels(['Legit', 'Fraud'])
axes[0].set_title('Purchase Value vs Class')

sns.boxplot(x='class', y='age', data=fraud_df, ax=axes[1],
            palette={0: 'steelblue', 1: 'tomato'})
axes[1].set_xticklabels(['Legit', 'Fraud'])
axes[1].set_title('Age vs Class')

fraud_rate_by_source = fraud_df.groupby('source')['class'].mean()
fraud_rate_by_source.plot.bar(ax=axes[2], color='coral', edgecolor='white')
axes[2].set_title('Fraud Rate by Source')
axes[2].set_ylabel('Fraud Rate')
axes[2].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fraud_rate_browser = fraud_df.groupby('browser')['class'].mean().sort_values(ascending=False)
fraud_rate_browser.plot.bar(ax=axes[0], color='slateblue', edgecolor='white')
axes[0].set_title('Fraud Rate by Browser')
axes[0].set_ylabel('Fraud Rate')
axes[0].tick_params(axis='x', rotation=30)

fraud_rate_sex = fraud_df.groupby('sex')['class'].mean()
fraud_rate_sex.plot.bar(ax=axes[1], color=['royalblue', 'orchid'])
axes[1].set_title('Fraud Rate by Sex')
axes[1].set_ylabel('Fraud Rate')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 5. Geolocation — IP to Country

In [ ]:
fraud_df = merge_ip_country(fraud_df, ip_df)

top_countries = fraud_df['country'].value_counts().head(15)
top_countries.plot.bar(figsize=(12, 4), color='teal', edgecolor='white')
plt.title('Top 15 Countries by Transaction Count')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
fraud_rate_country = (
    fraud_df.groupby('country')['class']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'fraud_rate', 'count': 'tx_count'})
    .query('tx_count >= 50')
    .sort_values('fraud_rate', ascending=False)
    .head(20)
)

fraud_rate_country['fraud_rate'].plot.bar(figsize=(12, 4), color='tomato', edgecolor='white')
plt.title('Fraud Rate by Country (min 50 transactions)')
plt.ylabel('Fraud Rate')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print(fraud_rate_country.head(10))

## 6. Data Cleaning

In [ ]:
print('Before cleaning:', fraud_df.shape)
fraud_df = drop_duplicates(fraud_df)
fraud_df = handle_missing(fraud_df, strategy='drop')
print('After cleaning:', fraud_df.shape)

# Save cleaned intermediate
fraud_df.to_csv('../data/processed/fraud_data_cleaned.csv', index=False)
print('Saved → data/processed/fraud_data_cleaned.csv')

## Summary

**Key Findings:**
- The dataset is heavily imbalanced — fraud cases make up ~9% of transactions.
- Short time-between-signup-and-purchase and high purchase values correlate with fraud.
- Certain countries and browsers show significantly higher fraud rates.
- IP-to-country enrichment provides additional geographic signal.

**Next step:** Feature Engineering notebook → `feature-engineering.ipynb`